# Step 1: - Import Libraries needed for the whole project and Load Dataset

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/jeisamathew/churn-analysis/Churn_Modelling_Cleaned.csv
/kaggle/input/notebooks/jeisamathew/churn-analysis/__results__.html
/kaggle/input/notebooks/jeisamathew/churn-analysis/__notebook__.ipynb
/kaggle/input/notebooks/jeisamathew/churn-analysis/__output__.json
/kaggle/input/notebooks/jeisamathew/churn-analysis/custom.css
/kaggle/input/notebooks/jeisamathew/churn-analysis/__results___files/__results___19_0.png
/kaggle/input/notebooks/jeisamathew/churn-analysis/__results___files/__results___22_0.png
/kaggle/input/notebooks/jeisamathew/churn-analysis/__results___files/__results___24_0.png
/kaggle/input/notebooks/jeisamathew/churn-analysis/__results___files/__results___21_0.png
/kaggle/input/notebooks/jeisamathew/churn-analysis/__results___files/__results___18_0.png


In [2]:
# Load the Kaggle dataset into a DataFrame
file_path = "/kaggle/input/notebooks/jeisamathew/churn-analysis/Churn_Modelling_Cleaned.csv"
df = pd.read_csv(file_path)


In [3]:
import sqlite3
conn = sqlite3.connect(":memory:")
df.to_sql("bank_customers", conn, index=False, if_exists="replace")

10000

In [4]:
#Verify that the data was imported correctly:
query = """
SELECT * FROM bank_customers LIMIT 10;
"""
result_df = pd.read_sql_query(query, conn)
result_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
6,822,France,Male,50,7,0.00,2,1,1,10062.80,0
7,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1
8,501,France,Male,44,4,142051.07,2,0,1,74940.50,0
9,684,France,Male,27,2,134603.88,1,1,1,71725.73,0


# Step 2 - Basic Business Queries

In [5]:
# Query 1: Total Customers
query_1 = """
SELECT COUNT(*) AS Total_Customers FROM bank_customers;
"""
result_df = pd.read_sql_query(query_1, conn)
result_df

,Total_Customers
0,10000


In [6]:
# Query 2: Total Customers Who Left
query_2 = """
SELECT COUNT(*) AS Churned_Customers FROM bank_customers WHERE Exited = 1;
"""
result_df = pd.read_sql_query(query_2, conn)
result_df

,Churned_Customers
0,2037


In [7]:
# Query 3: Churn Rate (Fixed Integer Division)
query_3 = """
SELECT ROUND((SUM(Exited) * 100.0) / COUNT(*), 2) AS Churn_Rate FROM bank_customers;
"""
result_df = pd.read_sql_query(query_3, conn)
result_df

,Churn_Rate
0,20.37


In [8]:
# Query 4: Customers by Country
query_4 = """
SELECT Geography, COUNT(*) AS Total_Customers FROM bank_customers GROUP BY Geography;
"""
df_geography = pd.read_sql_query(query_4, conn)
df_geography

,Geography,Total_Customers
0,France,5014
1,Germany,2509
2,Spain,2477


# Step 3 - Calculated Business Queries 

In [9]:
# Query 5: Churn by Country (Fixed Integer Division)
query_5 = """ 
SELECT Geography, COUNT(*) AS Customers, SUM(Exited) AS Churned, 
ROUND((SUM(Exited) * 100.0) / COUNT(*), 2) AS Churn_Rate
FROM bank_customers
GROUP BY Geography
ORDER BY Churn_Rate DESC;
"""

df_churn_by_country = pd.read_sql_query(query_5, conn)
df_churn_by_country


,Geography,Customers,Churned,Churn_Rate
0,Germany,2509,814,32.44
1,Spain,2477,413,16.67
2,France,5014,810,16.15


In [10]:
# Query 6: Churn by Gender (Fixed Integer Division)
query_6 = """
SELECT Gender,COUNT(*) AS Customers, SUM(Exited) AS Churned,
ROUND((SUM(Exited) * 100.0) / COUNT(*), 2) AS Churn_Rate
FROM bank_customers
GROUP BY Gender;
"""

df_gender_churn = pd.read_sql_query(query_6, conn)
df_gender_churn

,Gender,Customers,Churned,Churn_Rate
0,Female,4543,1139,25.07
1,Male,5457,898,16.46


In [11]:
# Query 7: Churn by Active Membership
query_7 = """
SELECT IsActiveMember, COUNT(*) AS Customers, SUM(Exited) AS Churned,
ROUND((SUM(Exited) * 100.0) / COUNT(*), 2) AS Churn_Rate
FROM bank_customers
GROUP BY IsActiveMember;
"""

df_active_churn = pd.read_sql_query(query_7, conn)
df_active_churn

,IsActiveMember,Customers,Churned,Churn_Rate
0,0,4849,1302,26.85
1,1,5151,735,14.27


In [12]:
# Query 8: Average Balance by Geography
query_8 = """
SELECT Geography,
       ROUND(AVG(Balance), 2) AS Avg_Balance
FROM bank_customers
GROUP BY Geography;
"""

df_avg_balance = pd.read_sql_query(query_8, conn)
df_avg_balance

,Geography,Avg_Balance
0,France,62092.64
1,Germany,119730.12
2,Spain,61818.15


In [13]:
# Query 9: Churn by Number of Products (Fixed Integer Division)
query_9 = """
SELECT NumOfProducts,
       COUNT(*) AS Customers,
       SUM(Exited) AS Churned,
       ROUND((SUM(Exited) * 100.0) / COUNT(*), 2) AS Churn_Rate
FROM bank_customers
GROUP BY NumOfProducts
ORDER BY NumOfProducts;
"""

df_product_churn = pd.read_sql_query(query_9, conn)
df_product_churn

,NumOfProducts,Customers,Churned,Churn_Rate
0,1,5084,1409,27.71
1,2,4590,348,7.58
2,3,266,220,82.71
3,4,60,60,100.00


# Step 4 - Advanced Calculated Business Queries

In [14]:
# Query 10: High-Value Customer Churn Financial Exposure
query_10 = """
SELECT 
    Geography,
    COUNT(CASE WHEN Balance > 100000 AND Exited = 1 THEN 1 END) AS High_Value_Churned_Count,
    ROUND(SUM(CASE WHEN Balance > 100000 AND Exited = 1 THEN Balance ELSE 0 END), 2) AS Total_Capital_Lost
FROM bank_customers
GROUP BY Geography
ORDER BY Total_Capital_Lost DESC;
"""

df_financial_exposure = pd.read_sql_query(query_10, conn)
df_financial_exposure

,Geography,High_Value_Churned_Count,Total_Capital_Lost
0,Germany,706,88262808.16
1,France,343,47984213.20
2,Spain,162,23242669.64


In [15]:
# Query 11: Credit Score Risk Cohort Analysis (Fixed Integer Division)
query_11 = """
SELECT 
    CASE 
        WHEN CreditScore >= 800 THEN 'Excellent (800+)'
        WHEN CreditScore >= 740 THEN 'Very Good (740-799)'
        WHEN CreditScore >= 670 THEN 'Good (670-739)'
        WHEN CreditScore >= 580 THEN 'Fair (580-669)'
        ELSE 'Poor (<580)'
    END AS Credit_Tier,
    COUNT(*) AS Total_Customers,
    SUM(Exited) AS Churned_Customers,
    ROUND((SUM(Exited) * 100.0) / COUNT(*), 2) AS Churn_Rate
FROM bank_customers
GROUP BY 1
ORDER BY Churn_Rate DESC;
"""

df_credit_tiers = pd.read_sql_query(query_11, conn)
df_credit_tiers

,Credit_Tier,Total_Customers,Churned_Customers,Churn_Rate
0,Poor (<580),2362,520,22.02
1,Very Good (740-799),1224,252,20.59
2,Fair (580-669),3331,685,20.56
3,Excellent (800+),655,128,19.54
4,Good (670-739),2428,452,18.62


In [16]:
# Query 12: Product Penetration & Financial Engagement Matrix (Fixed Integer Division)
query_12 = """
SELECT 
    HasCrCard,
    IsActiveMember,
    COUNT(*) AS Total_Customers,
    ROUND(AVG(NumOfProducts), 2) AS Avg_Products_Held,
    ROUND((SUM(Exited) * 100.0) / COUNT(*), 2) AS Churn_Rate
FROM bank_customers
GROUP BY HasCrCard, IsActiveMember
ORDER BY Churn_Rate DESC;
"""

df_engagement_matrix = pd.read_sql_query(query_12, conn)
df_engagement_matrix

,HasCrCard,IsActiveMember,Total_Customers,Avg_Products_Held,Churn_Rate
0,1,0,3448,1.52,27.32
1,0,0,1401,1.53,25.70
2,0,1,1544,1.53,16.39
3,1,1,3607,1.54,13.36


In [17]:
# Query 13: Tenure vs. Estimated Salary Demographics (Fixed Integer Division)
query_13 = """
SELECT 
    CASE 
        WHEN Tenure <= 2 THEN 'New Customer (0-2 Yrs)'
        WHEN Tenure <= 6 THEN 'Mid-Term Customer (3-6 Yrs)'
        ELSE 'Loyal Customer (7+ Yrs)'
    END AS Tenure_Cohort,
    ROUND(AVG(EstimatedSalary), 2) AS Avg_Salary,
    ROUND((SUM(Exited) * 100.0) / COUNT(*), 2) AS Churn_Rate
FROM bank_customers
GROUP BY 1;
"""

df_tenure_cohorts = pd.read_sql_query(query_13, conn)
df_tenure_cohorts


,Tenure_Cohort,Avg_Salary,Churn_Rate
0,Loyal Customer (7+ Yrs),100905.10,19.51
1,Mid-Term Customer (3-6 Yrs),99500.39,20.64
2,New Customer (0-2 Yrs),99878.64,21.15
